# Stylometric & Lexical Author Classification

This notebook focuses on **Stylometry** (the study of linguistic style) to improve author identification. Instead of focus on semantics (what word means), we'll focus on the specific lexical choices and stylistic fingerprints of **Edgar Allan Poe (EAP)**, **H.P. Lovecraft (HPL)**, and **Mary Shelley (MWS)**.

### Experiments:

1. **Word-level TF-IDF (Baseline)**
2. **Character-level TF-IDF** (Captures morphology, suffixes, and punctuation)
3. **POS-tagged TF-IDF** (Captures purely grammatical style)
4. **Custom Stylometric Features** (Sentence length, vocabulary complexity, function words)
5. **Ensemble Pipeline**


In [ ]:
import pandas as pd
import numpy as np
import nltk
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import optuna
import warnings

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
warnings.filterwarnings('ignore')

In [ ]:
# Load data
df = pd.read_csv('cleaned_data.csv')
df = df.dropna(subset=['cleaned_text'])

le = LabelEncoder()
df['target'] = le.fit_transform(df['author'])

X_train, X_val, y_train, y_val = train_test_split(
    df['cleaned_text'], df['target'], test_size=0.2, random_state=42
)

results = {}
print(f"Loaded {len(df)} samples with labels: {le.classes_}")

## 1. Word-level TF-IDF (Baseline)

Default TF-IDF often performs well because it captures the specific vocabulary frequency of each author.


In [ ]:
tfidf_word = TfidfVectorizer(ngram_range=(1, 1))
X_train_word = tfidf_word.fit_transform(X_train)
X_val_word = tfidf_word.transform(X_val)

lr_base = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_base.fit(X_train_word, y_train)
probs = lr_base.predict_proba(X_val_word)

results['Word TF-IDF Baseline'] = log_loss(y_val, probs)
print(f"Baseline Log-Loss: {results['Word TF-IDF Baseline']:.4f}")

## 2. Character-level TF-IDF

Character n-grams are excellent for author identification because they capture sub-word patterns, punctuation, and morphological markers that authors use habitually without realizing it.


In [ ]:
def objective_char(trial):
    n_range = trial.suggest_categorical('ngram_range', [(2, 4), (3, 5), (2, 5)])
    sublinear = trial.suggest_categorical('sublinear_tf', [True, False])
    
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=n_range, sublinear_tf=sublinear)
    X_tr = vectorizer.fit_transform(X_train)
    
    clf = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
    score = cross_val_score(clf, X_tr, y_train, cv=3, scoring='neg_log_loss').mean()
    return score

study_char = optuna.create_study(direction='maximize')
study_char.optimize(objective_char, n_trials=5)

params = study_char.best_params
tfidf_char = TfidfVectorizer(analyzer='char_wb', **params)
X_train_char = tfidf_char.fit_transform(X_train)
X_val_char = tfidf_char.transform(X_val)

lr_char = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_char.fit(X_train_char, y_train)
probs_char = lr_char.predict_proba(X_val_char)

results['Char TF-IDF (Optimized)'] = log_loss(y_val, probs_char)
print(f"Char-TFIDF Log-Loss: {results['Char TF-IDF (Optimized)']:.4f}")

## 3. Stylometric Feature Extraction

We create a custom class to extract features like word length, punctuation density, and function word ratios.


In [ ]:
class StylometricExtractor(BaseEstimator, TransformerMixin):
    def fit(self, x, y=None):
        return self
    
    def transform(self, posts):
        features = []
        for post in posts:
            words = post.split()
            features.append([
                len(post),                   # Raw length
                len(words),                  # Word count
                np.mean([len(w) for w in words]) if words else 0, # Avg word length
                len(set(words)) / len(words) if words else 0,     # Lexical diversity
                post.count(','),             # Punctuation habits
                post.count(';'),
                post.count('!'),
                len([w for w in words if w.lower() in ['the', 'and', 'of', 'in', 'to']]) / len(words) if words else 0 # Stopword density
            ])
        return np.array(features)

extractor = StylometricExtractor()
X_train_sty = extractor.transform(X_train)
X_val_sty = extractor.transform(X_val)

# Must scale custom features
scaler = StandardScaler()
X_train_sty = scaler.fit_transform(X_train_sty)
X_val_sty = scaler.transform(X_val_sty)

lr_sty = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_sty.fit(X_train_sty, y_train)
results['Stylometric Features Only'] = log_loss(y_val, lr_sty.predict_proba(X_val_sty))
print(f"Stylometric Logic-Loss: {results['Stylometric Features Only']:.4f}")

## 4. Part-of-Speech (POS) TF-IDF

This ignores the words and focuses on the sequence of grammar used.


In [ ]:
def get_pos_tags(text):
    tokens = nltk.word_tokenize(text)
    tags = nltk.pos_tag(tokens)
    return " ".join([tag for word, tag in tags])

X_train_pos = X_train.apply(get_pos_tags)
X_val_pos = X_val.apply(get_pos_tags)

tfidf_pos = TfidfVectorizer(ngram_range=(1, 3))
X_train_pos_vec = tfidf_pos.fit_transform(X_train_pos)
X_val_pos_vec = tfidf_pos.transform(X_val_pos)

lr_pos = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_pos.fit(X_train_pos_vec, y_train)
results['POS TF-IDF'] = log_loss(y_val, lr_pos.predict_proba(X_val_pos_vec))
print(f"POS-TFIDF Log-Loss: {results['POS TF-IDF']:.4f}")

## 5. Final Stylometric Union

We combine word-level, char-level, and manual stylometric features into one powerful feature set.


In [ ]:
combined_features = FeatureUnion([
    ('word_tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('char_tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True)),
    ('stylometry', Pipeline([
        ('extract', StylometricExtractor()),
        ('scale', StandardScaler())
    ]))
])

X_train_final = combined_features.fit_transform(X_train)
X_val_final = combined_features.transform(X_val)

lr_final = LogisticRegression(C=2.0, multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_final.fit(X_train_final, y_train)
final_probs = lr_final.predict_proba(X_val_final)

results['Combined Stylometric Pipeline'] = log_loss(y_val, final_probs)

print("\n--- Final Rankings (Log-Loss) ---")
for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name}: {score:.4f}")